<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 13 - Valoración, Interpretación y Despliegue de Modelos ML

**Módulo:** Sistemas de Big Data (SBD - 5074)

**Contenido:** 2.5 Fases de los modelos: métricas de evaluación, interpretabilidad, MLOps y despliegue en producción

---

## Objetivos
- Evaluar modelos usando métricas apropiadas según el negocio
- Interpretar resultados y comunicarlos al negocio
- Planificar el despliegue de modelos en producción
- Monitorizar modelos desplegados (data drift, performance drift)

## 1. Crear modelo de ejemplo

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
import plotly.express as px

# ========================================
# CREAR DATOS Y MODELO DE EJEMPLO
# ========================================
np.random.seed(42)
n_clientes = 1000

df = pd.DataFrame({
    'antiguedad_meses': np.random.randint(1, 72, n_clientes),
    'gasto_mensual': np.random.normal(100, 30, n_clientes).clip(20, 200),
    'num_productos': np.random.randint(1, 5, n_clientes),
    'llamadas_soporte': np.random.poisson(2, n_clientes),
    'satisfaccion': np.random.randint(1, 6, n_clientes),
    'tiene_contrato': np.random.choice([0, 1], n_clientes, p=[0.3, 0.7]),
})

# Crear variable objetivo (churn)
prob_churn = (
    (df['satisfaccion'] < 3) * 0.3 +
    (df['llamadas_soporte'] > 3) * 0.2 +
    (df['antiguedad_meses'] < 12) * 0.15 +
    (df['tiene_contrato'] == 0) * 0.2 +
    np.random.random(n_clientes) * 0.15
)
df['churn'] = (prob_churn > 0.5).astype(int)

# Preparar datos
X = df.drop('churn', axis=1)
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Entrenar modelo
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

print(f"Modelo entrenado con {len(X_train)} registros")
print(f"Accuracy en test: {model.score(X_test_scaled, y_test):.3f}")

## 2. Métricas de evaluación

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    roc_auc_score, roc_curve, precision_recall_curve
)
import plotly.graph_objects as go

# Predicciones
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

# Reporte de clasificación
print("=== REPORTE DE CLASIFICACIÓN ===")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

# AUC-ROC
auc = roc_auc_score(y_test, y_prob)
print(f"\nAUC-ROC: {auc:.3f}")

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
fig = px.imshow(cm, text_auto=True,
                labels=dict(x="Predicción", y="Real"),
                x=['No Churn', 'Churn'],
                y=['No Churn', 'Churn'],
                title='Matriz de Confusión',
                color_continuous_scale='Blues')
fig.show()

In [ ]:
# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f'ROC (AUC={auc:.3f})'))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random', line=dict(dash='dash')))
fig.update_layout(
    title='Curva ROC',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate'
)
fig.show()

## 3. Interpretabilidad de modelos

In [ ]:
# ========================================
# 1. IMPORTANCIA DE FEATURES (intrínseca)
# ========================================
print("=== IMPORTANCIA DE FEATURES ===")
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True)

print(importances)

fig = px.bar(importances, x='importance', y='feature', orientation='h',
             title='Importancia de Variables (Gini)')
fig.show()

In [ ]:
# ========================================
# 2. PERMUTATION IMPORTANCE (agnóstica)
# ========================================
print("\n=== PERMUTATION IMPORTANCE ===")
perm_importance = permutation_importance(model, X_test_scaled, y_test, n_repeats=10, random_state=42)

perm_df = pd.DataFrame({
    'feature': X.columns,
    'importance_mean': perm_importance.importances_mean,
    'importance_std': perm_importance.importances_std
}).sort_values('importance_mean', ascending=True)

print(perm_df)

fig = px.bar(perm_df, x='importance_mean', y='feature', orientation='h',
             error_x='importance_std',
             title='Permutation Importance')
fig.show()

## 4. Interpretación para negocio

In [ ]:
interpretacion = """
=== HALLAZGOS CLAVE ===

1. SATISFACCIÓN es el factor más importante
   → Clientes con satisfacción < 3 tienen 3x más probabilidad de abandonar
   
2. LLAMADAS A SOPORTE es el segundo factor
   → Más de 3 llamadas indica frustración y riesgo de churn
   
3. CONTRATO reduce significativamente el churn
   → Clientes sin contrato abandonan 2x más

=== RECOMENDACIONES ===

- Implementar programa de mejora de satisfacción
- Revisar procesos cuando un cliente llama > 2 veces
- Incentivar la firma de contratos con descuentos
"""
print(interpretacion)

## 5. Despliegue de Modelos (MLOps)

### Ciclo de vida MLOps

```
DESARROLLO           DESPLIEGUE          OPERACIÓN
┌─────────┐         ┌─────────┐         ┌─────────┐
│ Datos   │         │ Testing │         │ Monitor │
│ Feature │────────►│ CI/CD   │────────►│ Alertas │
│ Train   │         │ Deploy  │         │ Retrain │
└─────────┘         └─────────┘         └─────────┘
     ▲                                       │
     └───────────────────────────────────────┘
                  Feedback Loop
```

### Opciones de despliegue

| Tipo | Descripción | Latencia | Herramientas |
|------|-------------|----------|--------------|
| **Batch** | Predicciones periódicas programadas | Alta | Airflow, Cron |
| **API REST** | Predicciones bajo demanda | Baja | FastAPI, Flask |
| **Streaming** | Predicciones en tiempo real | Muy baja | Kafka, Flink |
| **Embebido** | Modelo en el dispositivo | Mínima | ONNX, TFLite |

### Monitorización post-despliegue

- **Data Drift**: los datos de entrada cambian respecto al entrenamiento
- **Concept Drift**: la relación entre features y target cambia con el tiempo
- **Performance Drift**: las métricas del modelo degradan gradualmente

### Ejemplo práctico: guardar y servir el modelo

In [ ]:
import joblib

# Guardar modelo y scaler
joblib.dump(model, 'modelo_churn.pkl')
joblib.dump(scaler, 'scaler_churn.pkl')

print("=== MODELO GUARDADO ===")
print("Archivos creados:")
print("  - modelo_churn.pkl")
print("  - scaler_churn.pkl")

# Simular predicción en producción
print("\n=== SIMULACIÓN DE PREDICCIÓN ===")

# Cargar modelo
modelo_prod = joblib.load('modelo_churn.pkl')
scaler_prod = joblib.load('scaler_churn.pkl')

# Nuevo cliente
nuevo_cliente = pd.DataFrame([{
    'antiguedad_meses': 6,
    'gasto_mensual': 80,
    'num_productos': 1,
    'llamadas_soporte': 5,
    'satisfaccion': 2,
    'tiene_contrato': 0
}])

# Predecir
cliente_scaled = scaler_prod.transform(nuevo_cliente)
prob = modelo_prod.predict_proba(cliente_scaled)[0][1]

print(f"\nCliente: {nuevo_cliente.to_dict('records')[0]}")
print(f"Probabilidad de churn: {prob:.1%}")
print(f"Recomendación: {'⚠️ ALTO RIESGO - Contactar' if prob > 0.5 else '✅ Bajo riesgo'}")

---

## Recursos adicionales

- [MLOps Principles](https://ml-ops.org/)
- [SHAP - Model Interpretability](https://shap.readthedocs.io/)
- [Scikit-learn Model Persistence](https://scikit-learn.org/stable/modules/model_persistence.html)